In [ ]:
import os
import csv
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
from PIL import Image
from tqdm import tqdm

import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
#-------------读取训练集,训练集地址已经设定好，下面这段不用修改------------------#
#-----Read the training set, the address of the training set has been set, and the following section does not need to be modified-------#
train_path = "/bohr/train-i5ob/v1"

In [ ]:
import os
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# 读取数据
def load_train_data(data_dir='./train/'):
    label_path = os.path.join(data_dir, 'train_labels.csv')
    image_dir = os.path.join(data_dir, 'train_images')

    data = []
    with open(label_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            sample_id = row['id']
            data.append({
                'id': sample_id,
                'image_path': os.path.join(image_dir, f'{sample_id}.png'),
                'sum': int(row['sum']),
                'product': int(row['product']),
            })

    print(f'Successfully loaded training records: {len(data)}')
    return data

class MNISTBaselineDataset(Dataset):
    def __init__(self, records, is_train):
        self.records = records
        self.is_train = is_train
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2)),
            transforms.ToTensor(),  # 转换为Tensor
        ])

    def __len__(self):
        return len(self.records)

    @staticmethod
    def load_image(image_path, is_train=False):
        image = Image.open(image_path).convert('L')
        if not is_train:
            image = np.array(image, dtype=np.float32) / 255.0
            image = (image - 0.1307) / 0.3081
        return image

    def __getitem__(self, idx):
        record = self.records[idx]
        image = self.load_image(record['image_path'], False)

        if self.is_train:
            # 应用数据增强
            # image = torch.tensor(np.asarray(image))
            # image = image.reshape(1, 28, 4, 28).swapaxes(1, 2).swapaxes(0, 1)
            # image = torch.concat([self.transform(image[i, :, :, :]) for i in range(4)], dim=0)
            # image = image.swapaxes(0, 1).swapaxes(1, 2).reshape(1, 28, 112)
            target = torch.tensor([record['sum'], record['product']], dtype=torch.long)
            return image, target
        
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        return image, 0

class MNISTTestDataset(Dataset):
    def __init__(self, image_dir):
        self.image_paths = sorted(str(path) for path in Path(image_dir).glob('*.png'))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        sample_id = Path(image_path).stem
        image = MNISTBaselineDataset.load_image(image_path)
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        return image, sample_id

def create_train_loader(batch_size=64, num_workers=1, data_dir='./train/', use_validation=False):
    records = load_train_data(data_dir)
    if use_validation:
        split_idx = int(len(records) * 0.8)
        train_records = records[:split_idx]
        val_records = records[split_idx:]
        train_set = MNISTBaselineDataset(train_records, True)
        return DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers), val_records
    else:
        train_set = MNISTBaselineDataset(records, True)
        return DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers), None

class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(256 * 3 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 10)  # 输出10个类别
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # 展平
        x = self.fc_layers(x)
        return x

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = MNISTModel()

    def forward(self, x):
        x = x.view(-1, 28, 4, 28).permute(0, 2, 1, 3).reshape(-1, 1, 28, 28) # split to four, equals batch*4
        x = torch.softmax(self.model(x), dim=-1)
        x = x.view(-1, 4, 10) # (batch, digits, classes)
        x = (x[:, 0, :].view(-1, 10, 1, 1, 1) * 
             x[:, 1, :].view(-1, 1, 10, 1, 1) * 
             x[:, 2, :].view(-1, 1, 1, 10, 1) * 
             x[:, 3, :].view(-1, 1, 1, 1, 10)).view(-1, 10000) # 10^4 digits, total 10000 classes
        return x

# 使用训练好的模型直接回归 sum 和 product
def predict(model, dataset, device='cpu', batch_size=64, num_workers=1):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    rows = []
    model.eval()
    with torch.no_grad():
        for images, sample_ids in tqdm(loader, desc='Predicting', leave=False):
            images = images.to(device)
            preds = model(images).argmax(dim=-1) 
            sum_preds = dsum[preds].cpu()
            product_preds = dprod[preds].cpu()

            for sample_id, sum_pred, product_pred in zip(sample_ids, sum_preds, product_preds):
                rows.append({
                    'id': sample_id,
                    'sum': int(sum_pred),
                    'product': int(product_pred),
                })
    return rows

# 训练模型
def train(model, train_loader, val_records, epochs, device='cpu'):
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-2, steps_per_epoch=len(train_loader), epochs=epochs)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for images, targets in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            preds = model(images)
            mask = (dsum == targets[:, [0]]) & (dprod == targets[:, [1]]) # sum==sum and prod==prod
            p = (preds * mask).sum(dim=-1) # the prob of crash can ignore and seems not critical
            loss = (-torch.log(p)).mean() 
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() * images.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        print(f'Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}')

        # 验证集推理
        if val_records is not None:
            val_dataset = MNISTBaselineDataset(val_records, is_train=False)
            val_predictions = predict(model, val_dataset, device=device)
            val_sum_acc = np.mean([1 if pred['sum'] == record['sum'] else 0 for pred, record in zip(val_predictions, val_records)])
            val_product_acc = np.mean([1 if pred['product'] == record['product'] else 0 for pred, record in zip(val_predictions, val_records)])
            print(f'Validation - Sum Accuracy: {val_sum_acc:.4f}, Product Accuracy: {val_product_acc:.4f}')

def set_random_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# 主程序
device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42
batch_size = 64
epochs = 30
USE_VALIDATION = False  # 设置为 True 使用验证集，设置为 False 使用全量训练

set_random_seed(seed)

# all classes
rng = torch.arange(10)
dsum = (rng.view(10, 1, 1, 1) + rng.view(1, 10, 1, 1) + rng.view(1, 1, 10, 1) + rng.view(1, 1, 1, 10)).view(-1).to(device)
dprod = (rng.view(10, 1, 1, 1) * rng.view(1, 10, 1, 1) * rng.view(1, 1, 10, 1) * rng.view(1, 1, 1, 10)).view(-1).to(device)

train_loader, val_records = create_train_loader(batch_size=batch_size, data_dir=train_path, use_validation=USE_VALIDATION)
model = Model()
train(model, train_loader, val_records, epochs=epochs, device=str(device))


In [20]:
# all classes 10**4
import torch
import numpy as np
A44=4*3*2*1
device="cpu"
rng = torch.arange(10)
dsum = (rng.view(10, 1, 1, 1) + rng.view(1, 10, 1, 1) + rng.view(1, 1, 10, 1) + rng.view(1, 1, 1, 10)).view(-1).to(device)
dprod = (rng.view(10, 1, 1, 1) * rng.view(1, 10, 1, 1) * rng.view(1, 1, 10, 1) * rng.view(1, 1, 1, 10)).view(-1).to(device)
a=len(set(dsum.detach().cpu().numpy().tolist()))
print(a/10000)
b=len(set(dprod.detach().cpu().numpy().tolist()))
print(b/10000)
cc = (torch.stack([dsum,dprod]).permute(1,0)).detach().cpu().numpy().tolist()
print(cc[0:5])
c2 = list(map(str,cc))
print(c2[0:5])
c=len(set(c2))
print(c/10000)

0.0037
0.0226
[[0, 0], [1, 0], [2, 0], [3, 0], [4, 0]]
['[0, 0]', '[1, 0]', '[2, 0]', '[3, 0]', '[4, 0]']
0.0494


In [ ]:
#-------------读取测试集---------------#“DATA_PATH”是测试集加密后的环境变量，按照如下方式可以在提交后，系统评分时访问测试集，但是选手无法直接下载
#----Read the testing set, “DATA_PATH” is an environment variable for the encrypted test set. After submission, you can access the test set for system scoring in the following manner, but the contestant cannot download it directly.-----#
if os.environ.get('DATA_PATH'):
    test_path = os.environ.get("DATA_PATH") + "/"
else:
    test_path = "./test/"
    print("Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")
    #Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象
    #When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.

In [ ]:
def save(rows, output_file):
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'sum', 'product'])
        writer.writeheader()
        writer.writerows(rows)
    print(f'Saved predictions to {output_file}')

def predict_and_save(model, image_dir, output_file, device='cpu', batch_size=64, num_workers=1):
    dataset = MNISTTestDataset(image_dir)
    rows = predict(model, dataset, device, batch_size, num_workers)
    save(rows, output_file)


val_dir = os.path.join(test_path, 'val')
test_dir = os.path.join(test_path, 'test')

for required_dir in [val_dir, test_dir]:
    if not os.path.isdir(required_dir):
        raise FileNotFoundError(f'Missing test directory: {required_dir}')

predict_and_save(model, val_dir, 'submission_val.csv', device=str(device), batch_size=batch_size)
predict_and_save(model, test_dir, 'submission_test.csv', device=str(device), batch_size=batch_size)

In [ ]:
import zipfile

# 定义要打包的文件和压缩文件名
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# 创建一个 zip 文件
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将文件添加到 zip 文件中
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} 创建成功!')